---
title: "Trotterized Hubbard Model"
title-block-banner: true
abstract: |
  Trotter error bound but more relevant for physics
---

<!---
todo: Check neel state set up
What is formular for order bound Trotter
--->

Generic resource estimation overestimates Trotter cost for structured Hamiltonians like Hubbard by X. We show this empirically using spectral norm, commutator bounds, and state-specific fidelity on a 2×2 model. We then demonstrate how Hubbard's bipartite and commutative structure can be exploited to reduce the resource estimate by factor Y

# Motivation for Trotterization
Why QSVT with QROM vs Trooterization

Hubbard is local and structured
QSVT/QROM is especially powerful when the Hamiltonian is given as a large irregular table: But the Hubbard model on a regular lattice is not an arbitrary table. It has simple repeated structure:

- nearest-neighbor hopping,
- osite interaction,
- mostly uniform coefficients,
- sparse local geometry.

In [37]:
# | code-fold: true
# | code-summary: "Import libraries"
import logging
from collections import defaultdict
from scipy import sparse
import matplotlib.pyplot as plt
import numpy as np
import pennylane as qp
import pennylane.estimator as qre
from pennylane.resource import SpectralNormError
import time as pytime
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)

# Setup {#sec-setup}

We create a $2 \times 2$ rectangle, with 4 sites, and 8 qubits. The notion is


| Qubit | Spin orbital |
|-------|--------------|
| 0     | site 0 ↑     |
| 1     | site 0 ↓     |
| 2     | site 1 ↑     |
| 3     | site 1 ↓     |
| 4     | site 2 ↑     |
| 5     | site 2 ↓     |
| 6     | site 3 ↑     |
| 7     | site 3 ↓     |



In [38]:
t = 1.0
U = 4.0
time = 3  # The time of evolution (t in exp(-iHt))

# number of Trotter steps, proposally not good to show the power of higher Trotter order
n_steps = int(time * 4)
num_steps = [10, 20, 40, 80, 120]

n_cells = [2, 2, 1]
orders = [1, 2, 4]
H = qp.spin.fermi_hubbard(
    lattice="cubic",
    n_cells=n_cells,
    hopping=t,
    coulomb=U,
    boundary_condition=False,
    mapping="jordan_wigner",
)
n_qubits = H.num_wires

This link is interesting https://fermi-hubbard-commutators.readthedocs.io/en/latest/commutator_bounds.html

# Naive Trotter error bound

How badly do standard Trotter error bounds overestimate cost for specific observables in Hubbard, and what's the practical implication for resource estimates?"
The spectral norm $\lVert {U_{exact} - U_{trotter}}\rVert_2$ measures the maximum possible deviation over all possible input states in the entire Hilbert space.

However in common application, these relevant states (e.g, low-energy excited states with specific symmetries) is in a much smaller space. Therefore it is not the best interest to tune the Trotter using the Spectral Norm.

The next natural question is for the Hubbard model, the most relavant states are? In this example we use half-filled Neel state ($\ket{0101...01}$), which is standard for studying Mott insulators.

Haar-random states are common in complexity proofs because they are uniformly distributed across the full Hilbert space, making them worst-case inputs. However, they have no physical relevance to Hubbard physics.

Compute grouping here  https://docs.pennylane.ai/en/stable/code/api/pennylane.ops.op_math.LinearCombination.html#pennylane.ops.op_math.LinearCombination.compute_grouping

In [39]:
# H.compute_grouping()  # compute the qubit-wise commuting groups!

# resources_exec = qre.estimate(executable_circuit)(grouped_hamiltonian, num_steps, order)

# resources_with_grouping = qre.estimate(
#     qre.TrotterPauli(kitaev_H_with_grouping, num_steps, order)
# )

# res = qre.estimate(circuit)(kitaev_H_with_grouping, num_steps, order)

## Spectral Norm {#sec-sec-spectral-norm}

Exact calculation, with such a small system, the Hamiltonian matrix is tractable ad diagonalizable. It stuggles at 16 qubits, which would result in $2^{16} \times 2^{16}$ matrix. However, due to the locality nature of Hubbard model, we can save a lot of memory using the sparse matrix

<!--
```
exact_op = qp.exp(H, -1j * time)
sparse_mat_exact_op = exact_op.sparse_matrix()
spectral_error = {}

for order in tqdm.tqdm(orders):
    approx_op = qp.TrotterProduct(H, n=n_steps, time=time, order=order)
    sparse_mat_approx_op = sparse.csr_matrix(qp.matrix(approx_op))
    spectral_error[order] =  sparse.linalg.svds(sparse_mat_exact_op - sparse_mat_approx_op, k=1, which='LM', return_singular_vectors=False)[0]

print(
    "\n".join(
        [f"Order: {k}, Spectral error: {v:5f}" for k, v in spectral_error.items()]
    )
)```
-->

In [40]:
exact_op = qp.exp(H, -1j * time)
spectral_error = {}

for order in orders:
    approx_op = qp.TrotterProduct(H, n=num_steps[0], time=time, order=order)
    error = SpectralNormError.get_error(exact_op, approx_op)
    spectral_error[order] = error

print(
    "\n".join(
        [f"Order: {k}, Spectral error: {v:5f}" for k, v in spectral_error.items()]
    )
)

Order: 1, Spectral error: 1.999668
Order: 2, Spectral error: 1.988046
Order: 4, Spectral error: 1.984760


## Childs Method

Since Calculating spectral norm is expensive because it requires diagonalizing the operators. This method promises better memory scalability, but it comes with a cost

In [41]:
# for order in orders[:2]:
#     op = qp.TrotterProduct(H, n=num_steps[0], time=time, order=order)
#     start = pytime.time()
#     one_norm_error_bound = op.error(method="one-norm-bound")
#     commutator_error_bound = op.error(method="commutator-bound")
#     print(f"Order {order}, Calculation time: {pytime.time() - start}")
#     print(f"One-norm bound: {one_norm_error_bound}")
#     print(f"Commutator bound: {commutator_error_bound}")
#     print()

With this tiny config, it costs a lot of time to compute, and the bound does not seem anywhere positive. They even contradicts each other. Suppose we want to estimate the order of the Trotter operation to get a reliable error, this really put us between a rock and a hard place,

For an explanation of the contradictions of the methods as well as the time cost, I invite readers to session @sec-sec-a1

So we spent a lot of time to calculate these bounds, but end up not gaining anything very informative about how to set the parameters. In larger system, the calculation of SpectralNorm error we did in @sec-sec-spectral-norm is imposible. 

Physicists only care for limited interesting states, and they live in a small manifold in the Hilbert space. Here we propose using state-aware error estimation

# Physics state-awared error bound estimation

- What are the super set of Neel state? I choose half filled state
- How does state-dependent error suppression affect the optimal Trotter order?
    - Perhaps with this we see 1-order is already very good
    - 2nd order meh
    - 4th order unncesscary
    - In the end "Do physically relevant states reduce the practical advantage of higher-order Trotter formulas in Hubbard simulation?"

## State initialization

Neel state ($\ket{0101...01}$) is the standard state for studying ferromagnetics problem. Using the notation in @sec-setup, we deduce that

In [42]:
def prepare_neel_state(wires):
    """Néel state: alternating up/down on bipartite lattice"""
    for site in range(wires // 2):
        if site % 2 == 0:
            qp.PauliX(wires=2 * site)
        else:
            qp.PauliX(wires=2 * site + 1)

## Hubbard Hamiltonian properties

We have two types of interaction in Hubbard: Hopping $T$ and on-site repulsion $U$

In [43]:
dev = qp.device("default.qubit")


@qp.qnode(dev)
def exact_circ(H, t, wires):
    """
    Simluate exact evolution
    """
    prepare_neel_state(wires)
    qp.exp(H, -1j * t)
    return qp.state()


@qp.qnode(dev)
def trotter_circ(H, num_steps, t, wires, order=1):
    """
    Because TrotterProduct evaluate exp(iHt), not exp(-iHt), we
    flip the H
    """
    prepare_neel_state(wires)
    qp.TrotterProduct(-H, n=num_steps, time=t, order=order)
    return qp.state()

In [44]:
resource_est = {}
for order in orders:
    resources_exec = qre.estimate(trotter_circ)(H, n_steps, time, n_qubits, order=order)
    resource_est[order] = resources_exec.gate_counts

In [ ]:
exact_state = exact_circ(H, time, n_qubits)
error_dicts = {}
state_fidelities = defaultdict(dict)

for order in orders:
    for n_step in num_steps:
        trotter_state = trotter_circ(H, n_step, time, n_qubits, order=order)
        state_fidelities[order][n_step] = qp.math.fidelity_statevector(exact_state, trotter_state)

In [ ]:
state_fidelities

In [ ]:
error_array = [spectral_error[order] for order in orders]
gates = [resource_est[order]["T"] for order in orders]

plt.plot(gates, error_array, label="Spectral Error", marker="o")
for o, xi, yi in zip(orders, gates, error_array):
    plt.annotate(
        f"Order = {o}",
        (xi, yi),
        textcoords="offset points",
        xytext=(0, 8),  # offset label above point
        ha='center'
    )

plt.ylabel("Error")
plt.xlabel("T gates")
plt.legend()
plt.show()

So we need to have `{python} gates[2]` T gates to make a simulation for `{python} time` seconds on a modest-sized Hubbard simulation.

In [ ]:
from scipy.linalg import expm
import numpy as np

H_mat = qp.matrix(H, wire_order=range(n_qubits))
U_exact = expm(-1j * time * H_mat)

neel = np.zeros(2**n_qubits, dtype=complex)
neel_index = sum(1 << i for i in [0, 3, 4, 7]) 
neel[neel_index] = 1.0

true_exact_state = U_exact @ neel
print(qp.math.fidelity_statevector(true_exact_state, exact_state))  # will be << 1

In [ ]:
state_fidelity_order_1_array = [state_fidelities[orders[0]][step] for step in num_steps]
state_fidelity_order_2_array = [state_fidelities[orders[1]][step] for step in num_steps]
state_fidelity_order_4_array = [state_fidelities[orders[2]][step] for step in num_steps]

plt.plot(num_steps, state_fidelity_order_1_array, label="Order 1")
plt.plot(num_steps, state_fidelity_order_2_array, label="Order 2")
plt.plot(num_steps, state_fidelity_order_4_array, label="Order 4")

plt.ylabel("State Fidelity")
plt.xlabel("Num steps")
plt.legend()
plt.show()

Higher orders, number of steps doesn't matter?
some

# Business output

If we increase the order based on the Spectral norm estimation, then the number of gates vs accuracy looks like this

But if we increase the order based on the state fidelity, then the number of gates vs accuracy looks like this


# Goal
- Practical resource estimation for early-to-intermediate fault-tolerant quantum simulation of strongly correlated materials. Choose 3D Hubbard model using PennyLane's Labs estimator
- Give systematic, empirical comparison of worst-case vs. observable-specific Trotter error bounds
- Able to distinguish asymptotic theory from realistic hardware constraints;
- Capable of producing technically honest estimates that are useful for strategic decision-making inside an FTQC organization.

# Appendix

## Appendix 1 {#sec-sec-a1}
<!--
This seems counter intuitive when commutator bound is more pessimistic than one-norm bound, and none of them are close to the spectral norm. higher order's error should scale with $O(\frac{t}{n})$. Order 2 is better algorithmically but has worse worst-case combinatorial bound.

One-norm bound is faster because it just adds every possible error

Commutator bound is tighter here it finds every pair $i, j$ such that [Hᵢ, Hⱼ] = 0. For the Hubbard model, all the interaction terms V = Σ U nᵢ↑nᵢ↓ commute with each other, so this pruning is substantial.
Why it runs out of memory at high orders — the combinatorial explosion:
For an order-p Trotter formula, the commutator bound requires computing nested commutators to depth p+1. With N Hamiltonian terms:
Trotter orderCommutator depth needed# evaluations1depth 2~N²2depth 3~N³4depth 5~N⁵-->